### 1 · Data Preprocessing — CMS3 Journey Logs to Retrieval-Ready Chunks

This notebook replaces the markdown-doc preprocessing flow with a **log-debugging preprocessing flow** for CMS3.

The goal is to turn raw CMS3 journey logs into **retrieval-friendly chunks** that an AI assistant can use to explain:
- why a customer moved through a certain path
- which condition triggered a route decision
- what happened before and after a specific step
- where failures or suspicious transitions occurred

### Pipeline
```
data/raw/cms3-logs.json
  → Load structured log events
  → Normalize fields and derive retrieval metadata
  → Group events by execution_id (journey run)
  → Build event chunks + execution summary chunks
  → Export to data/processed/cms3_log_chunks.json
```


In [5]:
import json
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from langchain_core.documents import Document

# Resolve notebook-relative paths once so every later step uses the same inputs/outputs.
NOTEBOOK_DIR = Path.cwd()
load_dotenv(dotenv_path=NOTEBOOK_DIR.parent / ".env", override=True)

RAW_FILE = NOTEBOOK_DIR.parent / "data/raw/cms3-logs.json"
PROCESSED_DIR = NOTEBOOK_DIR.parent / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_PATH = PROCESSED_DIR / "cms3_log_chunks.json"

# Load the raw CMS3 event list exactly as logged before any enrichment.
raw_logs = json.loads(RAW_FILE.read_text())
print(raw_logs[0])
print(f"Loaded {len(raw_logs)} raw CMS3 log events from {RAW_FILE}")
print(f"Top-level type: {type(raw_logs).__name__}")


{'id': 'log-b37e3643-f910-42c6-b9a4-ec06849e713f', 'timestamp': '2026-04-02T06:52:29.027Z', 'journey_id': 'ccflownew', 'execution_id': 'exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88', 'step_id': 'step-cf1a4acf-a876-4e7d-ba57-2d7eead56285', 'previous_step_id': 'step-582f5ca4-30f2-46a2-a251-75e973edcaac', 'step_order': 78, 'status': 'success', 'action': 'evaluate_ui_condition', 'page_path': '/ccflownew/quote/booking', 'customer_id': '7093495', 'input': {'context_snapshot': {}}, 'configuration': {'scope': 'config', 'item_id': 'moveStatus', 'condition': '{{isLaborJourney}} == true', 'condition_key': 'condition', 'config_path': 'condition', 'target': '../pricing'}, 'result': {'decision_result': False}, 'error_message': None, 'error_code': None, 'trace': {'environment': 'wip', 'app': 'cms3', 'release': 'development', 'session_id': 'session-b7a272d6-4fd6-4b3e-a27c-ba3fc40699b9', 'user_type': '10'}, 'level': 'info', 'eventType': 'ui_condition', 'message': 'UI condition evaluated', 'source'

## Step 1 — Normalize Log Events

Unlike markdown documents, logs already have structure.

So the preprocessing task here is not heading-aware splitting. It is:
- normalize optional fields
- derive stable metadata for retrieval
- turn each event into readable text for semantic search
- preserve identifiers for metadata filters (`execution_id`, `customer_id`, `journey_id`, `step_order`)


In [6]:
def normalize_log_event(log: dict) -> dict:
    """Convert a raw CMS3 log event into a retrieval-friendly record."""
    configuration = log.get("configuration") or {}
    result = log.get("result") or {}
    trace = log.get("trace") or {}

    decision_result = result.get("decision_result")
    target = configuration.get("target")
    condition = configuration.get("condition")

    # Build a readable text form so semantic retrieval can work on structured logs.
    lines = [
        f"timestamp: {log.get('timestamp', '')}",
        f"journey_id: {log.get('journey_id', '')}",
        f"execution_id: {log.get('execution_id', '')}",
        f"customer_id: {log.get('customer_id', 'unknown')}",
        f"step_order: {log.get('step_order', 'unknown')}",
        f"action: {log.get('action', '')}",
        f"event_type: {log.get('eventType', '')}",
        f"status: {log.get('status', '')}",
        f"level: {log.get('level', '')}",
        f"page_path: {log.get('page_path', '')}",
        f"source: {log.get('source', '')}",
        f"message: {log.get('message', '')}",
    ]

    if condition is not None:
        lines.append(f"condition: {condition}")
    if target is not None:
        lines.append(f"target: {target}")
    if decision_result is not None:
        lines.append(f"decision_result: {decision_result}")
    if result:
        lines.append(f"result_json: {json.dumps(result, sort_keys=True)}")
    if configuration:
        lines.append(f"configuration_json: {json.dumps(configuration, sort_keys=True)}")
    if log.get("error_code"):
        lines.append(f"error_code: {log['error_code']}")
    if log.get("error_message"):
        lines.append(f"error_message: {log['error_message']}")
    if trace:
        lines.append(f"trace_json: {json.dumps(trace, sort_keys=True)}")

    # Keep both structured fields for filtering and a text representation for retrieval.
    return {
        "id": log.get("id"),
        "timestamp": log.get("timestamp"),
        "journey_id": log.get("journey_id"),
        "execution_id": log.get("execution_id"),
        "customer_id": log.get("customer_id"),
        "step_id": log.get("step_id"),
        "previous_step_id": log.get("previous_step_id"),
        "step_order": log.get("step_order"),
        "status": log.get("status"),
        "level": log.get("level"),
        "action": log.get("action"),
        "event_type": log.get("eventType"),
        "page_path": log.get("page_path"),
        "source": log.get("source"),
        "message": log.get("message"),
        "target": target,
        "condition": condition,
        "decision_result": decision_result,
        "error_code": log.get("error_code"),
        "error_message": log.get("error_message"),
        "environment": trace.get("environment"),
        "release": trace.get("release"),
        "session_id": trace.get("session_id"),
        "event_text": "\n".join(lines),
        "raw": log,
    }


# Normalize every event into one consistent shape before grouping or chunking.
normalized_logs = [normalize_log_event(log) for log in raw_logs]

print("=== Sample normalized event ===")
pprint({
    key: normalized_logs[0][key]
    for key in [
        "execution_id",
        "customer_id",
        "step_order",
        "action",
        "page_path",
        "condition",
        "target",
        "decision_result",
    ]
})
print("\n=== Event text preview ===")
print(normalized_logs[0]["event_text"][:900])

=== Sample normalized event ===
{'action': 'evaluate_ui_condition',
 'condition': '{{isLaborJourney}} == true',
 'customer_id': '7093495',
 'decision_result': False,
 'execution_id': 'exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88',
 'page_path': '/ccflownew/quote/booking',
 'step_order': 78,
 'target': '../pricing'}

=== Event text preview ===
timestamp: 2026-04-02T06:52:29.027Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 78
action: evaluate_ui_condition
event_type: ui_condition
status: success
level: info
page_path: /ccflownew/quote/booking
source: ScriptedForm
message: UI condition evaluated
condition: {{isLaborJourney}} == true
target: ../pricing
decision_result: False
result_json: {"decision_result": false}
configuration_json: {"condition": "{{isLaborJourney}} == true", "condition_key": "condition", "config_path": "condition", "item_id": "moveStatus", "scope": "config", "target": "../pricing"}
trace_json

## Step 2 — Group Logs by Customer + Execution

For this use case, the main business identifier is the **customer ID**.

So instead of thinking only in terms of execution, we group logs around:
- `customer_id` as the primary application key
- `execution_id` as the journey-run key inside that customer context

This gives us a retrieval unit that matches how users will actually ask questions:
- "What happened for CID 7093495?"
- "Why did this customer go into this flow?"
- "Within this execution, which condition triggered the route?"


In [7]:
def build_group_key(event: dict) -> str:
    """Use customer_id as the main key and keep execution_id for run-level separation."""
    customer_id = event.get("customer_id") or "unknown_customer"
    execution_id = event.get("execution_id") or "unknown_execution"
    return f"cid:{customer_id} | exec:{execution_id}"


def group_logs_by_customer_and_execution(events: list[dict]) -> dict[str, list[dict]]:
    grouped: dict[str, list[dict]] = defaultdict(list)
    for event in events:
        grouped[build_group_key(event)].append(event)

    # Sorting by step_order reconstructs the journey in the order the user experienced it.
    for group_key, items in grouped.items():
        grouped[group_key] = sorted(
            items,
            key=lambda item: (item.get("step_order") or 0, item.get("timestamp") or ""),
        )
    return dict(grouped)


# Group around the business key (customer_id) while preserving execution boundaries.
execution_groups = group_logs_by_customer_and_execution(normalized_logs)

print(f"Customer/execution groups found: {len(execution_groups)}")
for group_key, items in execution_groups.items():
    customer_ids = sorted({item["customer_id"] for item in items if item.get("customer_id")})
    execution_ids = sorted({item["execution_id"] for item in items if item.get("execution_id")})
    pages = [item["page_path"] for item in items if item.get("action") == "route_entered"]
    print(
        f"- {group_key}: {len(items)} events | customers={customer_ids or ['unknown']} | executions={execution_ids or ['unknown']} | routes={len(pages)}"
    )


Customer/execution groups found: 2
- cid:7093495 | exec:exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88: 78 events | customers=['7093495'] | executions=['exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88'] | routes=22
- cid:unknown_customer | exec:exec-0f5de6ed-0424-47cf-893f-9f13158ccbfb: 7 events | customers=['unknown'] | executions=['exec-0f5de6ed-0424-47cf-893f-9f13158ccbfb'] | routes=2


## Step 3 — Build Retrieval-Ready Chunks

We create **two chunk types**:

1. `event`
   One chunk per log event. Good for precise retrieval like:
   - "Which condition evaluated to true?"
   - "What happened at step 77?"

2. `execution_summary`
   One chunk per execution. Good for broad retrieval like:
   - "What happened for CID 7093495?"
   - "Summarize this journey"
   - "Why did the flow end up on quote/booking?"

This is the log equivalent of semantic chunking for docs: small atomic evidence + larger contextual summary.


In [8]:
def build_event_document(event: dict) -> Document:
    # Event chunks preserve one atomic piece of evidence for precise retrieval.
    metadata = {
        "chunk_type": "event",
        "source": "cms3_logs",
        "journey_id": event.get("journey_id"),
        "execution_id": event.get("execution_id"),
        "customer_id": event.get("customer_id"),
        "step_id": event.get("step_id"),
        "previous_step_id": event.get("previous_step_id"),
        "step_order": event.get("step_order"),
        "status": event.get("status"),
        "level": event.get("level"),
        "action": event.get("action"),
        "event_type": event.get("event_type"),
        "page_path": event.get("page_path"),
        "target": event.get("target"),
        "decision_result": event.get("decision_result"),
        "error_code": event.get("error_code"),
        "environment": event.get("environment"),
        "release": event.get("release"),
    }
    return Document(page_content=event["event_text"], metadata=metadata)


def build_execution_summary_document(group_key: str, events: list[dict]) -> Document:
    # Route-entered events tell us the visible journey path through the app.
    routes = [
        event["page_path"]
        for event in events
        if event.get("action") == "route_entered" and event.get("page_path")
    ]
    unique_routes = list(dict.fromkeys(routes))

    # Pull condition checks into the summary because they often explain why a route changed.
    condition_hits = []
    for event in events:
        if event.get("action") == "evaluate_ui_condition":
            condition_hits.append(
                {
                    "step_order": event.get("step_order"),
                    "condition": event.get("condition"),
                    "decision_result": event.get("decision_result"),
                    "target": event.get("target"),
                    "page_path": event.get("page_path"),
                }
            )

    customer_ids = sorted({event["customer_id"] for event in events if event.get("customer_id")})
    execution_ids = sorted({event["execution_id"] for event in events if event.get("execution_id")})
    statuses = Counter(event.get("status") for event in events)
    actions = Counter(event.get("action") for event in events)

    # This summary chunk is intentionally broad so one retrieval hit can explain a full run.
    content_lines = [
        f"group_key: {group_key}",
        f"journey_id: {events[0].get('journey_id')}",
        f"customer_ids: {customer_ids or ['unknown']}",
        f"execution_ids: {execution_ids or ['unknown']}",
        f"event_count: {len(events)}",
        f"step_range: {events[0].get('step_order')} -> {events[-1].get('step_order')}",
        f"status_counts: {dict(statuses)}",
        f"top_actions: {dict(actions.most_common(8))}",
        f"route_path: {' -> '.join(unique_routes)}",
        "condition_checks:",
    ]

    for hit in condition_hits[:12]:
        content_lines.append(
            f"- step {hit['step_order']} | page={hit['page_path']} | condition={hit['condition']} | result={hit['decision_result']} | target={hit['target']}"
        )

    metadata = {
        "chunk_type": "execution_summary",
        "source": "cms3_logs",
        "group_key": group_key,
        "journey_id": events[0].get("journey_id"),
        "execution_id": execution_ids[0] if execution_ids else None,
        "customer_id": customer_ids[0] if customer_ids else None,
        "event_count": len(events),
        "status": "error" if any(event.get("status") != "success" for event in events) else "success",
        "first_step_order": events[0].get("step_order"),
        "last_step_order": events[-1].get("step_order"),
        "route_count": len(unique_routes),
    }
    return Document(page_content="\n".join(content_lines), metadata=metadata)


all_chunks: list[Document] = []
for group_key, events in execution_groups.items():
    all_chunks.append(build_execution_summary_document(group_key, events))
    all_chunks.extend(build_event_document(event) for event in events)

print(f"Built {len(all_chunks)} retrieval chunks")
print(Counter(chunk.metadata["chunk_type"] for chunk in all_chunks))


Built 87 retrieval chunks
Counter({'event': 85, 'execution_summary': 2})


## Step 4 — Preview the Output

We want to confirm that the exported chunks are both:
- readable by humans
- rich enough for retrieval filters


In [9]:
# Preview one summary chunk and one event chunk to validate readability and metadata shape.
summary_chunk = next(chunk for chunk in all_chunks if chunk.metadata["chunk_type"] == "execution_summary")
event_chunk = next(chunk for chunk in all_chunks if chunk.metadata["chunk_type"] == "event")

print("=" * 90)
print("EXECUTION SUMMARY METADATA")
print("=" * 90)
pprint(summary_chunk.metadata)
print("\nEXECUTION SUMMARY CONTENT")
print("=" * 90)
print(summary_chunk.page_content[:1800])

print("\n" + "=" * 90)
print("EVENT CHUNK METADATA")
print("=" * 90)
pprint(event_chunk.metadata)
print("\nEVENT CHUNK CONTENT")
print("=" * 90)
print(event_chunk.page_content[:1200])


EXECUTION SUMMARY METADATA
{'chunk_type': 'execution_summary',
 'customer_id': '7093495',
 'event_count': 78,
 'execution_id': 'exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88',
 'first_step_order': 1,
 'group_key': 'cid:7093495 | '
              'exec:exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88',
 'journey_id': 'ccflownew',
 'last_step_order': 78,
 'route_count': 22,
 'source': 'cms3_logs',
 'status': 'success'}

EXECUTION SUMMARY CONTENT
group_key: cid:7093495 | exec:exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
journey_id: ccflownew
customer_ids: ['7093495']
execution_ids: ['exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88']
event_count: 78
step_range: 1 -> 78
status_counts: {'success': 78}
top_actions: {'route_entered': 22, 'evaluate_ui_condition': 16, 'patch_context': 11, 'graphql_request': 9, 'navigate': 8, 'updateContext': 2, 'gql_get_available_agentsfrom_flex_query': 2, 'gql_create_lead_new': 2}
route_path: /ccflownew/returning-customer -> /ccflownew/check-skill-se

## Step 5 — Quality Checks

Before exporting, check whether the chunk set is shaped correctly.

Useful questions:
- Do we have both summary and event chunks?
- Are execution IDs preserved?
- Can we filter by customer / execution / action / page?
- Do we have any real failures in the sample?


In [10]:
# Basic stats here help verify the chunk set before we ingest it into embeddings.
chunk_sizes = [len(chunk.page_content) for chunk in all_chunks]
chunk_type_dist = Counter(chunk.metadata["chunk_type"] for chunk in all_chunks)
action_dist = Counter(
    chunk.metadata.get("action")
    for chunk in all_chunks
    if chunk.metadata["chunk_type"] == "event"
)
page_dist = Counter(
    chunk.metadata.get("page_path")
    for chunk in all_chunks
    if chunk.metadata["chunk_type"] == "event"
)

print("📊 Chunk Report")
print("=" * 60)
print(f"Total chunks:        {len(all_chunks)}")
print(f"Min chunk size:      {min(chunk_sizes)}")
print(f"Max chunk size:      {max(chunk_sizes)}")
print(f"Avg chunk size:      {sum(chunk_sizes) // len(chunk_sizes)}")
print(f"Execution summaries: {chunk_type_dist['execution_summary']}")
print(f"Event chunks:        {chunk_type_dist['event']}")
print()
print("📋 Event Coverage")
print("=" * 60)
print(f"Executions:          {len(execution_groups)}")
print(f"Customers present:   {len({chunk.metadata.get('customer_id') for chunk in all_chunks if chunk.metadata.get('customer_id')})}")
print(f"Failure events:      {sum(1 for event in normalized_logs if event.get('status') != 'success' or event.get('error_code') or event.get('error_message'))}")
print()
print("🔝 Top actions")
print("=" * 60)
for action, count in action_dist.most_common(10):
    print(f"{action:<35} {count:>3}")
print()
print("🔝 Top pages")
print("=" * 60)
for page, count in page_dist.most_common(10):
    print(f"{str(page):<35} {count:>3}")


📊 Chunk Report
Total chunks:        87
Min chunk size:      443
Max chunk size:      2320
Avg chunk size:      666
Execution summaries: 2
Event chunks:        85

📋 Event Coverage
Executions:          2
Customers present:   1
Failure events:      0

🔝 Top actions
route_entered                        24
evaluate_ui_condition                18
patch_context                        12
graphql_request                      10
navigate                              8
updateContext                         2
gql_get_available_agentsfrom_flex_query   2
gql_create_lead_new                   2
gql_lsa_lead_on_answer_call           1
gql_get_property_details_by_address   1

🔝 Top pages
/ccflownew/date                      17
/ccflownew/move-scope                 7
/ccflownew/items                      7
/ccflownew/quote/pricing              7
/ccflownew/junk-option                6
/ccflownew/name-and-phone             5
/ccflownew/quote/booking              5
/ccflownew/address                    4

## Step 6 — Export Processed Chunks

We export the chunks in the same general shape used elsewhere in the project:
```json
{
  "page_content": "...",
  "metadata": {...}
}
```

That keeps Notebook 2 (ingestion) simple.


In [11]:
# Export in the same page_content + metadata shape used by the rest of the project.
export_payload = [
    {
        "page_content": chunk.page_content,
        "metadata": chunk.metadata,
    }
    for chunk in all_chunks
]

EXPORT_PATH.write_text(json.dumps(export_payload, indent=2, default=str))
print(f"Exported {len(export_payload)} chunks to {EXPORT_PATH}")


Exported 87 chunks to /Users/sauravmajumdar/Developer/AI/move-mind-ai/data/processed/cms3_log_chunks.json


## Step 7 — Preview Retrieval Use Cases

The assistant should rely on **metadata filters first**, then semantic retrieval.

Examples:
- "For CID 7093495, what happened in this execution?"
- "Why did the flow go to `/ccflownew/quote/pricing`?"
- "Which UI condition evaluated to true on `/ccflownew/quote/booking`?"

This notebook prepares the data so later notebooks can do exactly that.


In [12]:
def filter_chunks(chunks: list[Document], **filters) -> list[Document]:
    # Lightweight helper to show how metadata-first retrieval will work later.
    results = []
    for chunk in chunks:
        if all(chunk.metadata.get(key) == value for key, value in filters.items()):
            results.append(chunk)
    return results


customer_chunks = filter_chunks(all_chunks, customer_id="7093495")
booking_conditions = [
    chunk for chunk in all_chunks
    if chunk.metadata.get("chunk_type") == "event"
    and chunk.metadata.get("action") == "evaluate_ui_condition"
    and chunk.metadata.get("page_path") == "/ccflownew/quote/booking"
]

print("Chunks for customer 7093495:", len(customer_chunks))
print("Condition events on /quote/booking:", len(booking_conditions))
print("\nSample booking condition event:\n")
print(booking_conditions[0].page_content[:1200])


Chunks for customer 7093495: 79
Condition events on /quote/booking: 4

Sample booking condition event:

timestamp: 2026-04-02T06:52:29.027Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 75
action: evaluate_ui_condition
event_type: ui_condition
status: success
level: info
page_path: /ccflownew/quote/booking
source: ScriptedForm
message: UI condition evaluated
condition: {{hasskillsetpricing}} == true
target: ../pricing
decision_result: False
result_json: {"decision_result": false}
configuration_json: {"condition": "{{hasskillsetpricing}} == true", "condition_key": "condition", "config_path": "conditions[0].condition", "item_id": "conditions[0].condition", "scope": "conditions", "target": "../pricing"}
trace_json: {"app": "cms3", "environment": "wip", "release": "development", "session_id": "session-b7a272d6-4fd6-4b3e-a27c-ba3fc40699b9", "user_type": "10"}


## What We Learned

This notebook now teaches the preprocessing stage for a **log-based AI debugger**:
- raw logs are already structured, so preprocessing is about **normalization and enrichment**, not heading splitting
- `execution_id` is the key unit for journey reconstruction
- event chunks provide atomic evidence
- execution summaries provide broad context
- metadata-first retrieval will matter more here than generic semantic search

Next step: ingest these chunks into the vector store and compare retrieval strategies for journey-debugging questions.


In [13]:
# Final checkpoint so the notebook ends with a clear output artifact.
print("Notebook 1 is now aligned to CMS3 log preprocessing.")
print(f"Processed chunk file ready at: {EXPORT_PATH}")


Notebook 1 is now aligned to CMS3 log preprocessing.
Processed chunk file ready at: /Users/sauravmajumdar/Developer/AI/move-mind-ai/data/processed/cms3_log_chunks.json
